In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [3]:
data=read.csv('Quarterly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [4]:
dim(data)

[1] 24000    27

In [5]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,4,1,4,0.9492015,1.440272e-05,-3.801739,-1.3040882,-0.4829440,0.4354882,0.3478599,⋯,-0.7896943,1.7947057,0.5618988,25,0.01836705,0.01836705,0.01836705,0,0,0
2,4,1,4,0.9332456,2.024615e-05,-3.528016,-1.5740960,-0.3277471,0.3143940,0.1373249,⋯,-0.6669261,0.7095412,0.5378848,25,0.02149177,0.02149177,0.02149177,0,0,0
3,4,1,4,0.9753689,3.173153e-06,-4.202105,-1.3102058,-0.3300377,0.2486809,0.1689488,⋯,-0.7152289,0.8909252,0.6050959,25,0.01739883,0.01739883,0.01739883,0,0,0
4,4,1,4,0.9597504,7.178017e-06,-4.163362,-0.7087874,-0.2787477,0.4831910,0.1164472,⋯,-0.6048727,0.6906562,0.6227230,25,0.01743579,0.01743579,0.01743579,0,0,0
5,4,1,4,0.9770473,1.682990e-07,-5.666607,0.6009075,-0.5379779,0.5748836,0.9582223,⋯,-0.2911521,1.9555069,0.7981635,59,0.01771545,0.01771545,0.01771545,0,0,0
6,4,1,4,0.9884024,1.093815e-07,4.960507,-1.0952883,-0.1131941,0.3831215,0.6847558,⋯,-0.5923500,2.3289573,0.6350911,59,0.01755309,0.01755309,0.01755309,0,0,0


In [6]:
dlist= load('Quarterly_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [7]:
dim(MASE)

[1] 24000     5     4

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
nanum

[1]   158   159   160   208   237   244   255   259   266   301   303   314
  [13]   322   327   330   340  1081  1201  1558  1564  1576  1579  1582  1585
  [25]  1586  1615  1626  1627  1629  1630  1639  1641  1644  1655  1656  1659
  [37]  1704  1709  1737  1744  1745  1751  1754  1756  1759  1763  1767  1769
  [49]  1773  1776  1778  1779  1784  1787  1788  1789  1792  1793  1794  1802
  [61]  1828  1855  1867  1881  1890  1898  1899  1902  1921  1933  1934  1935
  [73]  1937  1973  1974  2001  2023  2035  2053  2065  2071  2077  2091  2196
  [85]  2210  2217  2230  2233  2239  2283  2284  2300  2306  2323  2344  2399
  [97]  2436  2671  2676  2678  2703  2743  2781  3073  3149  3234  3235  3326
 [109]  3327  3328  3329  3330  3331  3332  3333  3334  3335  3336  3337  3448
 [121]  3449  3450  3451  3452  3453  3454  3455  3456  3514  3631  3634  3659
 [133]  3670  3672  3678  3700  3709  3747  3748  3749  3750  3751  3752  3753
 [145]  3754  3755  3756  3757  3758  3759  3760  3761  4121  4122  4123  4124
 [157]  4125  4126  4127  4128  4129  4130  4131  4132  4133  4134  4135  4136
 [169]  4158  4159  4160  4161  4162  4163  4164  4165  4166  4167  4168  4169
 [181]  4170  4171  4172  4173  4174  4175  4176  4177  4178  4179  4180  4181
 [193]  4182  4183  4184  4185  4186  4187  4188  4189  4190  4191  4192  4193
 [205]  4194  4195  4196  4197  4198  4199  4200  4201  4202  4203  4204  4411
 [217]  4412  4413  4415  4416  4417  4423  4442  4443  4444  4445  4446  4447
 [229]  4448  4449  4450  4451  4452  4453  4454  4455  4456  4457  4458  4459
 [241]  4460  4461  4462  4463  4464  4465  4466  4467  4468  4469  4470  4471
 [253]  4472  4473  4474  4475  4476  4477  4478  4519  4520  4523  4524  4525
 [265]  4526  4527  4528  4529  4530  4531  4532  4533  4534  4535  4536  4537
 [277]  4538  4539  4540  4541  4542  4543  4544  4545  4546  4547  4548  4549
 [289]  4550  4551  4552  4553  4554  4555  4556  4799  4813  4817  4823  4826
 [301]  4829  4831  4832  4836  4839  4840  4900  4924  4926  4946  4947  4952
 [313]  4953  4967  5032  5103  5104  5105  5117  5119  5120  5169  5178  5179
 [325]  5180  5181  5182  5183  5184  5185  5186  5187  5188  5189  5190  5307
 [337]  5308  5309  5514  5515  5516  5525  5528  5529  5530  5531  5532  5533
 [349]  5555  5559  5560  5576  5583  5584  5585  5589  5593  5594  5595  5596
 [361]  5600  5601  5602  5608  5617  5619  5631  5632  5634  5653  5656  5657
 [373]  5664  5665  5679  5680  5684  5685  5686  5687  5688  5689  5691  5703
 [385]  5704  5713  5764  5770  5774  5781  5793  5795  5806  5807  5809  5824
 [397]  5825  5828  5842  5844  5854  5856  5871  5872  5886  5887  5901  5911
 [409]  5916  5917  5922  5924  5934  5935  5936  5937  5938  5939  6968  6969
 [421]  6970  6971  6972  6973  6974  6975  6976  6977  6978  6979  6980  6981
 [433]  6982  6983  6984  6985  6986  6987  6988  6989  6990  6991  6992  6993
 [445]  6994  6995  6996  6997  6998  6999  7000  7001  7002  7003  7004  7005
 [457]  7006  7007  7008  7021  7024  7026  7038  7039  7040  7041  7042  7043
 [469]  7044  7045  7046  7047  7048  7049  7050  7051  7052  7053  7054  7055
 [481]  7056  7057  7058  7059  7060  7061  7062  7063  7064  7065  7066  7067
 [493]  7068  7069  7070  7083  7101  7104  7110  7121  7122  7173  7175  7197
 [505]  7465  7466  7467  7468  7469  7470  7471  7472  7473  7474  7475  7476
 [517]  7477  7478  7479  7480  7481  7482  7483  7484  7485  7486  7487  7488
 [529]  7489  7490  7491  7492  7493  7494  7495  7496  7497  7498  7499  7500
 [541]  7501  7502  7503  7504  7505  7506  7507  7508  7509  7510  7511  7512
 [553]  7513  7514  7515  7516  7517  7518  7519  7520  7521  7522  7532  7534
 [565]  7536  7537  7538  7539  7540  7543  7544  7547  7550  7551  7552  7555
 [577]  7556  7557  7559  7564  7565  7567  7568  7569  7575  7578  7583  7584
 [589]  7585  7590  7591  7592  7593  7594  7595  7597  7600  7601  7602  7605
 [601]  7606  7607  7609  7631  7638  7639  7640  764

In [11]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
   0    1    2    3    4 
3503 3076 3947 6060 7414 

In [13]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [14]:
realbestmean

2
2
2
1
0
3
2
2
2
0
0


In [15]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,4,1,4,0.9492015,1.440272e-05,-3.801739,-1.3040882,-0.4829440,0.4354882,0.3478599,⋯,-0.7896943,1.7947057,0.5618988,25,0.01836705,0.01836705,0.01836705,0,0,0
2,4,1,4,0.9332456,2.024615e-05,-3.528016,-1.5740960,-0.3277471,0.3143940,0.1373249,⋯,-0.6669261,0.7095412,0.5378848,25,0.02149177,0.02149177,0.02149177,0,0,0
3,4,1,4,0.9753689,3.173153e-06,-4.202105,-1.3102058,-0.3300377,0.2486809,0.1689488,⋯,-0.7152289,0.8909252,0.6050959,25,0.01739883,0.01739883,0.01739883,0,0,0
4,4,1,4,0.9597504,7.178017e-06,-4.163362,-0.7087874,-0.2787477,0.4831910,0.1164472,⋯,-0.6048727,0.6906562,0.6227230,25,0.01743579,0.01743579,0.01743579,0,0,0
5,4,1,4,0.9770473,1.682990e-07,-5.666607,0.6009075,-0.5379779,0.5748836,0.9582223,⋯,-0.2911521,1.9555069,0.7981635,59,0.01771545,0.01771545,0.01771545,0,0,0
6,4,1,4,0.9884024,1.093815e-07,4.960507,-1.0952883,-0.1131941,0.3831215,0.6847558,⋯,-0.5923500,2.3289573,0.6350911,59,0.01755309,0.01755309,0.01755309,0,0,0


In [16]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [17]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [18]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,4,1,4,0.9492015,1.440272e-05,-3.801739,-1.3040882,-0.4829440,0.4354882,0.3478599,⋯,-0.7896943,1.7947057,0.5618988,25,0.01836705,0.01836705,0.01836705,0,0,0
2,4,1,4,0.9332456,2.024615e-05,-3.528016,-1.5740960,-0.3277471,0.3143940,0.1373249,⋯,-0.6669261,0.7095412,0.5378848,25,0.02149177,0.02149177,0.02149177,0,0,0
3,4,1,4,0.9753689,3.173153e-06,-4.202105,-1.3102058,-0.3300377,0.2486809,0.1689488,⋯,-0.7152289,0.8909252,0.6050959,25,0.01739883,0.01739883,0.01739883,0,0,0
4,4,1,4,0.9597504,7.178017e-06,-4.163362,-0.7087874,-0.2787477,0.4831910,0.1164472,⋯,-0.6048727,0.6906562,0.6227230,25,0.01743579,0.01743579,0.01743579,0,0,0
5,4,1,4,0.9770473,1.682990e-07,-5.666607,0.6009075,-0.5379779,0.5748836,0.9582223,⋯,-0.2911521,1.9555069,0.7981635,59,0.01771545,0.01771545,0.01771545,0,0,0
6,4,1,4,0.9884024,1.093815e-07,4.960507,-1.0952883,-0.1131941,0.3831215,0.6847558,⋯,-0.5923500,2.3289573,0.6350911,59,0.01755309,0.01755309,0.01755309,0,0,0


In [19]:
end_time = Sys.time()

In [20]:
time_matrix[1,]=end_time-start_time

In [21]:
end_time-start_time

Time difference of 8.286007 secs

## Target the interval where the actual error is minimum

In [22]:
start_time = Sys.time()

In [23]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [24]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [25]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:1.921655 
[2]	train-rmse:1.623835 
[3]	train-rmse:1.449043 
[4]	train-rmse:1.352082 
[5]	train-rmse:1.294304 
[6]	train-rmse:1.260646 
[7]	train-rmse:1.240497 
[8]	train-rmse:1.228017 
[9]	train-rmse:1.218087 
[10]	train-rmse:1.211537 
[11]	train-rmse:1.205602 
[12]	train-rmse:1.199236 
[13]	train-rmse:1.196396 
[14]	train-rmse:1.191373 
[15]	train-rmse:1.188459 
[16]	train-rmse:1.182525 
[17]	train-rmse:1.181139 
[18]	train-rmse:1.177912 
[19]	train-rmse:1.173245 
[20]	train-rmse:1.171881 
[21]	train-rmse:1.169006 
[22]	train-rmse:1.164384 
[23]	train-rmse:1.161411 
[24]	train-rmse:1.158681 
[25]	train-rmse:1.154426 
[26]	train-rmse:1.149255 
[27]	train-rmse:1.143185 
[28]	train-rmse:1.138924 
[29]	train-rmse:1.134757 
[30]	train-rmse:1.131593 
[31]	train-rmse:1.128289 
[32]	train-rmse:1.125054 
[33]	train-rmse:1.122628 
[34]	train-rmse:1.118683 
[35]	train-rmse:1.111193 
[36]	train-rmse:1.106551 
[37]	train-rmse:1.103415 
[38]	train-rmse:1.099871 
[39]	train-rmse:1.094

In [26]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.534000 
[2]	train-mlogloss:1.480562 
[3]	train-mlogloss:1.440894 
[4]	train-mlogloss:1.408224 
[5]	train-mlogloss:1.380780 
[6]	train-mlogloss:1.359510 
[7]	train-mlogloss:1.339360 
[8]	train-mlogloss:1.322181 
[9]	train-mlogloss:1.305054 
[10]	train-mlogloss:1.290748 
[11]	train-mlogloss:1.277521 
[12]	train-mlogloss:1.265152 
[13]	train-mlogloss:1.256132 
[14]	train-mlogloss:1.247558 
[15]	train-mlogloss:1.239966 
[16]	train-mlogloss:1.229089 
[17]	train-mlogloss:1.221593 
[18]	train-mlogloss:1.211621 
[19]	train-mlogloss:1.202557 
[20]	train-mlogloss:1.194965 
[21]	train-mlogloss:1.187396 
[22]	train-mlogloss:1.179862 
[23]	train-mlogloss:1.172479 
[24]	train-mlogloss:1.165012 
[25]	train-mlogloss:1.161617 
[26]	train-mlogloss:1.153289 
[27]	train-mlogloss:1.149708 
[28]	train-mlogloss:1.141740 
[29]	train-mlogloss:1.135824 
[30]	train-mlogloss:1.127760 
[31]	train-mlogloss:1.120399 
[32]	train-mlogloss:1.115214 
[33]	train-mlogloss:1.109838 
[34]	train-mlogloss

In [27]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4811
[LightGBM] [Info] Number of data points in the train set: 16927, number of used features: 21
[LightGBM] [Info] Start training from score 2.457435


In [28]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017352 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4811
[LightGBM] [Info] Number of data points in the train set: 16927, number of used features: 21
[LightGBM] [Info] Start training from score -1.936502
[LightGBM] [Info] Start training from score -2.058802
[LightGBM] [Info] Start training from score -1.803585
[LightGBM] [Info] Start training from score -1.372390
[LightGBM] [Info] Start training from score -1.171253


In [29]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [30]:
start_time = Sys.time()

In [31]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [32]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [33]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.678534 
[2]	train-rmse:1.467918 
[3]	train-rmse:1.348601 
[4]	train-rmse:1.278984 
[5]	train-rmse:1.238840 
[6]	train-rmse:1.214818 
[7]	train-rmse:1.199831 
[8]	train-rmse:1.185376 
[9]	train-rmse:1.177762 
[10]	train-rmse:1.171192 
[11]	train-rmse:1.161893 
[12]	train-rmse:1.157028 
[13]	train-rmse:1.147895 
[14]	train-rmse:1.145244 
[15]	train-rmse:1.142812 
[16]	train-rmse:1.141130 
[17]	train-rmse:1.136624 
[18]	train-rmse:1.133881 
[19]	train-rmse:1.132015 
[20]	train-rmse:1.126269 
[21]	train-rmse:1.121698 
[22]	train-rmse:1.115438 
[23]	train-rmse:1.110164 
[24]	train-rmse:1.107718 
[25]	train-rmse:1.105993 
[26]	train-rmse:1.101001 
[27]	train-rmse:1.094798 
[28]	train-rmse:1.093149 
[29]	train-rmse:1.087175 
[30]	train-rmse:1.082206 
[31]	train-rmse:1.076830 
[32]	train-rmse:1.073872 
[33]	train-rmse:1.071635 
[34]	train-rmse:1.067139 
[35]	train-rmse:1.060829 
[36]	train-rmse:1.057066 
[37]	train-rmse:1.054920 
[38]	train-rmse:1.052407 
[39]	train-rmse:1.050

In [34]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.547209 
[2]	train-mlogloss:1.500310 
[3]	train-mlogloss:1.464830 
[4]	train-mlogloss:1.433249 
[5]	train-mlogloss:1.406502 
[6]	train-mlogloss:1.383358 
[7]	train-mlogloss:1.364090 
[8]	train-mlogloss:1.347497 
[9]	train-mlogloss:1.330028 
[10]	train-mlogloss:1.316457 
[11]	train-mlogloss:1.303076 
[12]	train-mlogloss:1.290711 
[13]	train-mlogloss:1.279098 
[14]	train-mlogloss:1.270127 
[15]	train-mlogloss:1.259051 
[16]	train-mlogloss:1.250475 
[17]	train-mlogloss:1.239814 
[18]	train-mlogloss:1.232032 
[19]	train-mlogloss:1.223921 
[20]	train-mlogloss:1.218477 
[21]	train-mlogloss:1.209782 
[22]	train-mlogloss:1.204139 
[23]	train-mlogloss:1.199058 
[24]	train-mlogloss:1.192379 
[25]	train-mlogloss:1.184425 
[26]	train-mlogloss:1.176348 
[27]	train-mlogloss:1.170544 
[28]	train-mlogloss:1.164477 
[29]	train-mlogloss:1.157021 
[30]	train-mlogloss:1.151899 
[31]	train-mlogloss:1.147092 
[32]	train-mlogloss:1.143071 
[33]	train-mlogloss:1.135474 
[34]	train-mlogloss

In [35]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011275 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4811
[LightGBM] [Info] Number of data points in the train set: 16927, number of used features: 21
[LightGBM] [Info] Start training from score 1.997164


In [36]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4811
[LightGBM] [Info] Number of data points in the train set: 16927, number of used features: 21
[LightGBM] [Info] Start training from score -1.585044
[LightGBM] [Info] Start training from score -1.776691
[LightGBM] [Info] Start training from score -1.561962
[LightGBM] [Info] Start training from score -1.362419
[LightGBM] [Info] Start training from score -1.831961


In [37]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [38]:
end_time-start_time

Time difference of 3.54173 mins

## predict

In [39]:
start_time = Sys.time()

In [40]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [41]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [42]:
xgbregmin

[1]  1.678847075  2.196492195  2.039880276  1.631919146  1.130262017
    [6]  2.234442234  2.126114845  2.578284502  2.777132988  2.360685349
   [11]  1.682183146  2.401282549  2.378574133  2.215666056  1.475599408
   [16]  2.815427780  2.399661064  1.811874747  2.199201822  2.204082966
   [21]  2.125024557  3.154262781  3.232300758  1.692758322  2.916904926
   [26]  2.019187927  1.509433985  3.067646265  1.957460403  1.148835063
   [31]  2.349027157  2.173642635  2.155700207  2.757719755  2.189349174
   [36]  3.188462257  2.609694958  2.777057171  2.036886215  1.686612368
   [41]  1.244007945  1.915075421  2.461247921  2.062518120  1.462778091
   [46]  2.877420187  3.306163311  2.317509651  2.351312876  2.440857410
   [51]  2.260694981  2.439066410  2.217546225  2.264739275  2.285696507
   [56]  1.940544724  1.814587951  2.391540289  2.575481892  3.114255190
   [61]  3.255158186  2.631958723  2.995078087  3.303005695  1.856225252
   [66]  3.689611435  2.753265381  1.856667519  3.024650335  2.929448605
   [71]  1.475754976  3.492564678  2.184723854  2.661376715  1.067652464
   [76]  2.780360222  2.592822790  2.225185871  1.992012978  0.935863733
   [81]  2.990128279  1.530846119  1.148543835  3.524352789  3.719002247
   [86]  1.165849209  2.228123665  1.355759978  1.815532088  4.116433620
   [91]  2.082741499  0.133624762  1.074004292  3.711094856  3.080430269
   [96]  2.979250431  3.889401197  1.866292596  1.740680337  1.941419601
  [101]  1.764821172  2.227084875  1.004523635  3.488415241  1.166857839
  [106]  2.434157133  1.590731621  1.095494270  2.822778702  1.682954192
  [111]  1.573654175  1.504765511  1.885696411  1.910945058  2.060121775
  [116]  2.581387758  2.091354609  1.935963154  1.273712039  3.083402157
  [121]  0.707977951  2.111901522  2.871163845  0.842371166  2.593249559
  [126]  2.438488483  1.497356415  2.723469496  2.237867832  2.501115084
  [131]  3.159856558  3.256636143  3.699127436  2.322740316  2.704833508
  [136]  1.464628816  2.184133530  2.883663893  2.198277473  3.122625589
  [141]  2.552573681  3.463275671  3.653491020  3.877525568  2.178106070
  [146]  3.110078096  3.057132244  2.708776236  2.819230318  2.197490931
  [151]  2.346106291  1.593912601  1.875863671  2.630438566  2.289174318
  [156]  2.104814053  3.286248922  2.718425512  0.310539752  0.783407509
  [161]  0.935169220  1.372096062  1.499936461  0.254771858  1.006731987
  [166]  2.300774336  1.561592698  2.215716124  1.951647758  1.116350174
  [171]  3.182897329  0.233355865  0.606018007  1.752729893  1.259726524
  [176]  1.407585740  2.379384995  0.410636365  2.228292465  1.103285551
  [181]  0.540655136  1.086917520  1.930748701  0.641667366  0.742329836
  [186]  0.365209341  1.703612447  1.441409945  1.257508159  1.203784347
  [191]  1.586059690  1.087009430  1.219825864  1.225595236  2.120196581
  [196]  1.724875808  0.207838327  0.966975272  2.884591579  2.992269754
  [201]  1.532767892  2.828263521  1.598246813  0.626477420  1.518035650
  [206]  2.412412167  2.628880501  1.550460219  3.079468250  2.841194153
  [211]  2.949361801  2.975143194  3.294548512  2.673100948  1.710000277
  [216]  1.905509949  2.288256884  1.082389116  3.071243763  3.241003990
  [221]  3.351795435  3.628067493  3.679234743  2.730233669  3.136658192
  [226]  1.545715451  2.816159487  3.182578087  2.039482594  0.240420237
  [231]  3.433687449  1.025833607  3.773173809  2.710556507  3.320072889
  [236]  3.616073847  0.502567649  2.620654821  2.887051821  2.831637383
  [241]  3.071104050  2.700939894  1.253702164  2.094345093  3.585831642
  [246]  2.745975018  1.050502896  1.661511421  3.029112339  3.574868441
  [251]  2.380366087  2.641563416  3.610710382  2.700274467  2.939825773
  [256]  2.720953703  3.198694706  2.231788158  1.859796047  2.050660849
  [261]  2.886522055  2.987713814  3.288528204  3.438424110  1.811218143
  [266]  2.845331192  2.437747478  3.317812443  2.700883865  2.724989653
  [271]  2.531485319  1.770603538  3.796587944  2.443406582  2

In [43]:
xgbclsmin

[1] 2 2 2 2 0 4 4 4 4 4 0 1 0 3 4 4 4 2 2 4 0 4 4 1 4 2 3 4 3 0 3 4 0 4 2 4
   [37] 4 3 4 0 0 4 0 4 1 4 4 4 3 3 1 3 1 0 0 4 0 3 4 4 3 2 0 3 1 4 2 2 4 4 2 4
   [73] 2 4 0 4 4 4 0 0 3 1 4 4 4 0 4 3 0 4 1 0 0 4 4 4 4 3 2 1 2 4 0 4 0 3 1 1
  [109] 3 0 4 4 2 0 0 3 0 2 0 4 0 4 4 0 4 0 0 3 3 4 3 4 4 2 4 0 2 3 4 3 2 4 4 4
  [145] 2 4 4 3 4 2 0 1 2 3 4 3 4 3 0 0 0 2 0 0 0 4 1 4 4 2 4 0 0 1 0 0 3 0 4 1
  [181] 0 0 0 0 0 0 1 0 2 0 0 1 0 0 2 2 0 1 3 4 1 4 4 4 0 3 4 1 4 4 4 2 4 4 1 3
  [217] 4 0 3 4 4 4 4 3 4 4 4 4 2 0 4 1 4 4 4 4 0 3 4 4 4 4 1 4 4 4 1 1 4 4 4 4
  [253] 4 4 3 4 4 4 3 4 4 4 4 4 4 4 3 4 4 4 3 4 4 3 3 4 4 0 4 3 3 4 4 4 4 3 4 4
  [289] 3 4 4 1 4 4 3 3 0 3 2 3 3 4 3 4 3 4 4 2 0 4 3 4 0 0 4 0 0 4 3 4 3 0 4 3
  [325] 0 4 3 1 4 0 3 4 4 4 3 4 4 4 4 2 4 4 3 1 4 4 3 0 0 4 4 1 3 3 1 2 4 4 1 2
  [361] 2 1 0 3 1 4 4 2 4 4 4 4 1 4 4 2 3 3 2 2 2 0 4 0 1 0 2 0 4 4 0 2 3 3 4 0
  [397] 0 0 4 2 4 2 2 4 4 0 4 4 4 4 4 0 4 4 4 0 0 4 3 0 4 4 3 4 3 4 1 4 3 1 1 4
  [433] 2 4 4 4 4 4 4 4 4 2 1 2 1 4 0 4 2 4 0 4 4 4 4 2 4 3 1 4 2 4 4 4 1 2 4 3
  [469] 4 2 4 1 4 4 4 4 4 4 4 4 4 3 4 1 1 4 4 4 4 4 4 4 4 0 0 3 0 4 4 1 4 4 1 1
  [505] 4 4 4 1 4 4 0 4 4 0 4 4 1 0 2 1 0 4 4 4 1 2 4 4 4 4 4 4 4 4 4 4 4 2 4 3
  [541] 4 3 2 4 4 2 4 4 3 4 2 4 1 4 0 4 4 4 1 2 4 2 1 4 4 3 4 2 4 4 2 4 4 3 4 3
  [577] 0 1 1 3 4 0 2 3 4 4 4 4 4 4 1 4 2 4 4 4 4 4 1 0 4 4 2 4 0 4 4 3 4 4 0 4
  [613] 0 3 0 0 4 3 2 2 0 4 1 4 0 4 0 4 4 4 1 3 4 4 4 1 1 4 4 4 2 4 4 1 1 4 0 0
  [649] 3 4 1 4 0 4 4 4 4 4 4 0 4 2 4 0 0 3 4 4 4 4 1 2 1 0 1 4 1 1 0 1 4 1 2 4
  [685] 4 4 4 1 3 3 4 1 4 4 4 4 4 3 1 4 1 0 0 0 4 4 2 4 4 4 0 4 4 0 4 4 4 1 0 2
  [721] 4 4 4 1 1 4 4 4 4 4 3 4 1 4 1 3 2 4 4 4 4 4 4 0 4 1 4 1 0 4 2 2 3 1 3 0
  [757] 2 4 3 0 1 4 4 0 4 3 4 4 3 4 4 2 4 4 4 4 4 1 4 1 2 4 4 4 4 2 0 0 3 4 1 1
  [793] 4 1 3 1 1 4 3 4 1 4 0 4 1 4 1 0 0 1 0 0 1 1 4 1 4 1 0 3 4 3 1 4 4 4 3 4
  [829] 1 4 4 2 2 4 0 0 1 4 4 4 0 0 4 0 4 4 2 4 2 4 2 4 4 4 4 4 1 2 2 4 4 4 2 4
  [865] 4 4 4 4 4 1 4 4 2 3 4 4 4 0 0 4 1 4 0 0 4 4 0 4 4 4 4 4 4 3 3 1 0 0 4 4
  [901] 4 4 0 3 4 4 3 1 4 4 2 4 0 1 4 1 3 3 4 3 4 3 4 4 1 4 1 4 3 4 1 1 3 0 4 4
  [937] 4 4 1 4 4 1 0 4 4 4 4 1 2 4 4 4 4 4 4 1 1 4 4 4 4 3 0 1 2 4 0 4 4 3 0 4
  [973] 4 4 4 4 0 4 1 2 4 4 4 1 4 4 1 0 4 4 2 0 4 4 2 4 3 4 4 4 4 4 4 4 2 0 1 4
 [1009] 4 4 0 2 4 0 4 4 4 4 4 4 4 4 1 0 4 0 0 1 4 0 4 4 2 4 1 0 3 4 4 4 4 0 4 4
 [1045] 3 3 4 4 4 0 4 2 4 3 2 2 2 3 4 2 3 4 2 4 4 0 4 0 4 2 0 1 1 3 3 2 2 4 1 4
 [1081] 4 4 0 4 3 4 4 4 4 0 0 3 0 0 0 4 4 2 0 3 2 4 4 4 0 0 4 4 1 4 0 4 3 4 2 4
 [1117] 4 4 4 0 4 4 1 0 1 0 4 2 4 4 4 4 2 4 4 1 1 4 1 4 3 3 0 0 0 3 3 0 4 4 4 1
 [1153] 2 4 1 1 4 4 1 2 1 1 1 2 4 1 4 1 3 4 4 4 4 4 3 4 3 4 4 4 4 4 3 3 4 4 4 4
 [1189] 1 4 4 1 4 4 1 0 4 1 4 4 3 4 1 4 3 3 4 1 4 4 3 3 0 1 0 4 3 3 4 4 4 4 1 4
 [1225] 1 4 1 4 4 1 0 4 0 0 0 0 0 4 4 0 4 4 4 3 3 1 4 4 4 1 2 4 4 1 4 4 0 4 4 3
 [1261] 4 4 4 4 2 3 1 2 4 1 1 4 4 2 4 4 4 3 4 4 4 2 4 4 3 3 4 4 2 4 3 4 4 4 4 4
 [1297] 4 1 4 2 4 0 4 4 1 4 2 2 4 1 0 0 4 0 4 3 1 1 4 0 4 2 4 4 4 4 4 0 0 0 4 1
 [1333] 4 4 4 4 2 1 4 4 3 4 4 4 4 4 3 4 3 4 4 2 4 4 3 4 4 1 3 1 1 2 3 4 4 3 2 4
 [1369] 2 2 3 4 4 0 4 4 4 4 4 4 4 4 4 4 4 4 1 4 4 4 0 4 1 4 3 4 2 0 2 1 2 4 4 1
 [1405] 4 4 2 3 4 1 4 3 4 1 4 4 4 4 4 4 4 4 1 1 3 2 2 4 2 4 4 4 4 0 4 3 3 4 4 4
 [1441] 4 4 4 2 2 4 4 4 4 4 4 4 1 4 3 4 4 4 4 0 2 1 4 4 2 4 4 1 4 1 4 4 4 1 2 4
 [1477] 4 4 4 4 3 0 0 0 0 1 3 0 2 3 0 3 3 4 1 3 4 0 4 4 4 4 1 4 1 2 4 0 4 4 4 0
 [1513] 0 4 4 0 0 2 0 4 4 4 4 0 4 4 1 4 0 4 0 0 4 0 1 1 4 2 4 0 2 3 1 0 1 3 1 4
 [1549] 3 0 1 2 4 0 2 0 0 1 4 2 1 1 4 0 4 4 1 3 2 3 2 4 1 1 4 2 4 1 1 0 1 1 2 4
 [1585] 4 1 4 4 0 2 0 4 4 1 2 3 1 0 4 3 3 3 1 2 0 3 1 4 4 0 0 4 0 4 3 2 2 1 3 3
 [1621] 2 0 3 3 2 3 2 4 3 0 4 4 2 2 4 4 4 2 1 4 3 3 4 2 4 3 0 1 3 3 0 0 4 3 3 1
 [1657] 0 1 3 4 4 3 3 3 4 3 4 4 3 0 4 2 4 1 0 1 3 1 4 1 4 0 4 3 4 4 3 4 0 0 4 4
 [1693] 4 1 1 1 0 4 3 0 1 3 4 3 4 4 0 4 3 1 0 3 4 1 0 0 4 0 1 1 4 1 4 3 1 2 2 1
 [1729] 1 1 4 4 4 0 4 4 2 0 4 3 2 2 4 0 2 4 4 4 4 4 3 3 4 3 4 3 3 4 1 4 3 4 1 4
 [1765] 4 4 0 3 3 4 2 4 3 4 4 3 3 3 3 4 4 4 4 0 3 4 3 1 3 3 3 2 3 3 3 4 4 3 4 4
 [18

In [44]:
lgbregmin

[1] 1.5071196 1.5962776 1.3542608 1.3923528 1.3634821 1.9738661 1.7420033
    [8] 2.4350817 1.9443495 2.1961087 1.1683025 1.7698839 2.0855295 2.3924090
   [15] 1.2137819 2.4293169 2.2621307 1.8451393 2.3006538 2.4833108 1.9669308
   [22] 2.3735616 2.9280350 1.8707888 2.0037804 2.1419652 1.7564257 2.1254973
   [29] 1.8281363 1.6866725 2.1223759 2.0981537 2.0435297 2.2531961 2.0182287
   [36] 2.2257892 2.1627540 2.4297902 2.1037573 1.6467009 1.8878266 2.0390519
   [43] 1.8428070 1.6460643 1.8618469 2.3186601 2.5171046 2.5340655 2.5068256
   [50] 2.2283387 2.4367191 2.3737835 2.3641556 2.0883703 2.0498935 1.9122741
   [57] 2.0949738 2.3600807 2.3200089 2.3368757 3.3922239 2.4766603 2.1180485
   [64] 3.3742149 1.7935970 3.3124460 2.5777385 2.0827550 2.5733976 2.8671783
   [71] 2.0445978 2.5793460 2.1708728 2.2006147 1.7072125 2.8675903 2.9772696
   [78] 3.0510971 2.3859410 3.3839772 2.7460643 1.8999476 1.6916014 3.6324519
   [85] 3.4707061 1.3309432 1.5911932 1.6175165 1.3175728 4.0828058 2.1301931
   [92] 0.6135951 1.0901790 3.4246571 3.8182657 2.9785423 3.2259890 1.8317731
   [99] 2.1799545 2.0789224 2.1414146 2.3123372 1.6555366 3.4049340 1.9454315
  [106] 2.6207479 1.8315361 1.6210236 3.2739511 1.5780898 2.1449758 2.0380169
  [113] 1.9257041 1.4273104 2.4367118 2.3441498 1.3249293 2.1684201 1.4620084
  [120] 2.6728970 1.6533070 1.7513165 2.2700918 1.6533070 2.7083208 1.8906282
  [127] 1.2458141 1.8555714 1.7546268 2.5210363 2.8651663 3.2865564 3.0494279
  [134] 2.5949247 2.1580815 1.8144578 2.4805486 3.1143396 2.3788057 3.1907236
  [141] 2.9761667 3.2736306 3.5920338 4.0645991 2.8168554 3.6231506 3.8606118
  [148] 2.7334386 2.8757196 2.1605676 1.7618157 1.3220199 2.3716972 2.7590020
  [155] 2.4316508 1.6330569 3.4147874 1.9498621 1.1601244 1.1225125 1.1066373
  [162] 1.4370032 1.6890713 0.7447765 1.1274882 1.9478114 1.3633691 1.9204090
  [169] 1.6304411 1.3524171 1.9243302 0.7273357 1.0652753 1.8228621 1.2456602
  [176] 1.4518025 1.4538523 0.8362821 2.0963866 1.0080791 0.6713447 1.0636077
  [183] 1.4791881 1.1904491 1.1870045 1.0396947 1.5040639 1.5731836 1.5695534
  [190] 0.9156188 1.3266457 1.0065876 1.3000065 1.3015447 1.5745860 1.6006694
  [197] 1.0580745 1.2710175 2.9791978 2.4800998 1.2695098 3.1668101 1.7378904
  [204] 1.9972535 2.4137444 2.5468252 3.6042256 1.4717315 2.7922169 2.9527206
  [211] 2.5884987 2.6014007 2.8436361 2.5801614 2.2264987 2.1089608 2.6993234
  [218] 1.7838152 3.1471393 3.0604249 3.2684433 3.5369816 3.4988059 3.0295049
  [225] 3.2537491 2.6970069 2.6917588 3.2800497 2.0942106 1.1849705 3.4723669
  [232] 1.4135250 3.7344672 2.2220955 3.3689865 3.4092348 1.0485190 2.0861813
  [239] 3.0201815 3.0113225 2.7322654 3.0003280 1.4637285 2.3295719 3.5091967
  [246] 2.7771097 1.5125480 1.7979500 2.9244196 3.3376114 2.5830055 1.9590032
  [253] 3.4725091 2.5490989 2.4087200 2.5711912 3.2949951 2.5100231 2.1616551
  [260] 2.1054265 2.8502586 2.5728537 3.2014625 3.3674811 2.5243434 2.7466387
  [267] 2.4336797 3.3893902 2.7731332 2.4174577 2.2428176 2.3223462 3.3351089
  [274] 2.2425977 2.1771939 2.6625136 3.1143125 1.3428138 2.0590778 2.3458747
  [281] 2.5946322 3.2588139 3.3265113 2.5997520 3.3913030 2.7113203 3.1706057
  [288] 3.9192899 2.1604858 2.6157786 3.6256481 1.7137460 2.2795961 3.0491078
  [295] 2.5586885 3.0085127 0.9608162 2.7740825 2.6143476 2.9267116 2.8524701
  [302] 3.1445379 2.4205774 2.9758319 2.3608300 2.3383360 2.1505163 2.5134292
  [309] 1.3514803 1.8157278 2.6388773 2.4659212 2.4440415 0.8848403 2.9461590
  [316] 2.3721997 2.4019264 3.0048582 2.7405241 2.7863188 2.3918034 1.3554962
  [323] 2.5593133 2.7928437 2.1904576 2.4473851 2.6175116 2.2495910 3.2811248
  [330] 1.6692440 2.4155666 3.4720835 3.5410358 3.4962322 3.2960773 2.2419209
  [337] 3.3382727 2.3790716 1.7665544 1.6184873 3.9824351 3.2787805 3.1998518
  [344] 2.1800052 2.7876074 2.1719141 1.9743818 1.8225846 2.2152057 2.0983104
  [351] 2.5235472 1.2751133 2.2358039 2.2358039 1.6192778 2.6458066 2.2554983
  [358] 2.1509408 1.487707

In [45]:
lgbclsmin

0.17213631,0.23578240,0.48562799,0.09869372,0.007759576
0.28779525,0.15844205,0.33630051,0.20003993,0.017422245
0.16686318,0.15671453,0.57916893,0.09025152,0.007001848
0.20263797,0.14865945,0.53019196,0.08813424,0.030376390
0.46342306,0.14797212,0.14301295,0.11437646,0.131215409
0.17649596,0.21378529,0.15050249,0.25941275,0.199803511
0.21634225,0.14265185,0.30874899,0.13745839,0.194798520
0.20660424,0.18877573,0.13433325,0.17582104,0.294465737
0.30788284,0.13558092,0.20204487,0.10589043,0.248600944
0.17365901,0.16443626,0.18487890,0.20597654,0.271049292
0.46857582,0.12096755,0.13539181,0.11009021,0.164974613


In [46]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [47]:
lgbclsminr

4
3
0
2
4
4
2
4
0
0
4


In [48]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [49]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [50]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [51]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
2,1.678847,4,1.507120
2,2.196492,3,1.596278
2,2.039880,0,1.354261
2,1.631919,2,1.392353
0,1.130262,4,1.363482
4,2.234442,4,1.973866


In [52]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [53]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
1,1.405556,4,1.069478
2,1.583426,4,1.256274
2,1.417093,2,1.160063
1,0.679006,3,1.199624
0,0.529897,3,1.039940
0,1.784399,4,1.722238


In [54]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
1,1.405556,4,1.069478
2,1.583426,4,1.256274
2,1.417093,2,1.160063
1,0.679006,3,1.199624
0,0.529897,3,1.039940
0,1.784399,4,1.722238


In [55]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [56]:
result_list=list(preallmin,preallmean,time_matrix)

In [57]:
save(result_list, file = "Quarterly_nnetar_opt_pre_result.RData")

In [58]:
time_matrix

user_time,system_time,elapsed_time
8.286007,8.286007,8.286007
9.302883,9.302883,9.302883
3.541730,3.541730,3.541730
1.551108,1.551108,1.551108
